In [20]:
import sys
import os
import importlib  # <-- ADDED THIS LINE

# Get the absolute path to the project root (Capstone)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the 'src' folder to Python's system path
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Project root added to path: {project_root}")
print("Ready to import from 'src' folder.")

Project root added to path: c:\Users\Caroline-PC\OneDrive\Desktop\Capstone
Ready to import from 'src' folder.


In [21]:
# Import the module names
import data_processing
import graph_construction

# --- ADD THESE LINES TO FORCE A RELOAD ---
importlib.reload(data_processing)
importlib.reload(graph_construction)
# --- END OF NEW LINES ---

# Now we can safely import the functions from the reloaded modules
from data_processing import find_all_contract_paths, process_contract_pdf
from graph_construction import process_contract_to_graph

# 1. Find all your contracts
pdf_files = find_all_contract_paths()

if pdf_files:
    print(f"Found {len(pdf_files)} total contracts.")
    
    # 2. Get the first one to process
    first_contract_path = pdf_files[0]
    
    # 3. Run the FULL pipeline (Processing + Graphing)
    G = process_contract_to_graph(first_contract_path, process_contract_pdf)
    
    # 4. See the result!
    print("\n--- PIPELINE COMPLETE ---")
    print(G) # This will now show the new edge count
    
else:
    print("No PDF files found. Check your 'Contracts' folder setup.")

Loading embedding model (this may take a moment)...
Embedding model loaded.
Found 242 total contracts.

--- Processing: contract_146.pdf ---
Found 21 nodes (clauses/paragraphs).
Generating 21 embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.25it/s]

Embeddings generated.
Added 20 STRUCTURAL edges.
Calculating semantic similarities...
Added 2 new SEMANTIC edges.
Graph built: 21 nodes, 22 total edges.

--- PIPELINE COMPLETE ---
Graph with 21 nodes and 22 edges


In [22]:
print(f"--- Analyzing Graph for Contract: {first_contract_path} ---")

semantic_edges = []
for u, v, data in G.edges(data=True):
    if data['type'] == 'SEMANTIC':
        semantic_edges.append((u, v, data['score']))

if not semantic_edges:
    print("\nNo semantic edges found. (Try lowering the SIMILARITY_THRESHOLD in graph_construction.py)")
else:
    print(f"\nFound {len(semantic_edges)} SEMANTIC EDGES:")
    
    # Sort by score, highest first
    semantic_edges.sort(key=lambda x: x[2], reverse=True)
    
    for i, (node1, node2, score) in enumerate(semantic_edges[:5]): # Print top 5
        print(f"\n--- Top Match #{i+1} (Score: {score:.4f}) ---")
        
        # Get the text for each node
        node1_text = G.nodes[node1]['content']
        node2_text = G.nodes[node2]['content']
        
        print(f"  NODE {node1} TEXT: {node1_text[:80]}...")
        print(f"  NODE {node2} TEXT: {node2_text[:80]}...")

--- Analyzing Graph for Contract: C:\Users\Caroline-PC\OneDrive\Desktop\Capstone\Contracts\Contract Risk Analyzer\Alphabet\contract_146.pdf ---

Found 2 SEMANTIC EDGES:

--- Top Match #1 (Score: 0.8250) ---
  NODE 3 TEXT: I, Sundar Pichai, certify that:...
  NODE 17 TEXT: Sundar Pichai...

--- Top Match #2 (Score: 0.8035) ---
  NODE 9 TEXT: (b) Designed such internal control over financial reporting, or caused such inte...
  NODE 13 TEXT: (a) All significant deficiencies and material weaknesses in the design or operat...


In [24]:
# --- Cell 4: Test Stage 3 (Model Definition) ---

import torch  # <-- THIS IS THE FIX for the NameError

# Reload our modules to get the new function
import graph_construction
import model_definition
importlib.reload(graph_construction)
importlib.reload(model_definition)

from graph_construction import convert_nx_to_pyg_data
from model_definition import GATModel

# 1. Convert our graph to the format PyG needs
# G should be in memory from running Cell 2
pyg_data = convert_nx_to_pyg_data(G)

print(f"\nPyG Data Object:\n{pyg_data}")

# 2. Initialize our GNN Model
model = GATModel(in_features=384, hidden_features=64, num_classes=3)
print(f"\nInitialized GNN Model:\n{model}")

# 3. Perform a "test forward pass"
model.eval()
with torch.no_grad():
    output = model(pyg_data)

print("\n--- TEST FORWARD PASS SUCCESSFUL ---")
print(f"Model output shape: {output.shape}")
print(f"(Shape is [num_nodes, num_classes] = [21, 3])")
print("\nFirst 5 predictions (log-probabilities):")
print(output[:5])

Loading embedding model (this may take a moment)...
Embedding model loaded.

Converted NetworkX graph to PyTorch Geometric data object.

PyG Data Object:
Data(x=[21, 384], edge_index=[2, 44], y=[21])

Initialized GNN Model:
GATModel(
  (conv1): GATConv(384, 64, heads=4)
  (conv2): GATConv(256, 3, heads=1)
)

--- TEST FORWARD PASS SUCCESSFUL ---
Model output shape: torch.Size([21, 3])
(Shape is [num_nodes, num_classes] = [21, 3])

First 5 predictions (log-probabilities):
tensor([[-1.0019, -1.1431, -1.1584],
        [-1.0032, -1.1478, -1.1521],
        [-1.0103, -1.1355, -1.1564],
        [-1.0479, -1.1054, -1.1448],
        [-1.0600, -1.0706, -1.1687]])


In [38]:
# --- Cell 5: Run the Full Train/Test Pipeline & SAVE Results [DEBUG] ---

import graph_construction
import model_definition
import train
import torch # Need torch to save

# Reload all modules
importlib.reload(graph_construction)
importlib.reload(model_definition)
importlib.reload(train)

# Run the pipeline and get the data for plotting
trained_model, train_data, test_data, test_preds, losses = train.run_training_pipeline()

# --- [DEBUG] Print status before saving ---
print("\n--- Checking variables before saving ---")
print(f"train_data is valid: {train_data is not None and hasattr(train_data, 'y')}")
print(f"test_data is valid: {test_data is not None and hasattr(test_data, 'y')}")
print(f"test_preds is valid: {test_preds is not None and isinstance(test_preds, torch.Tensor)}")
print(f"losses list has items: {bool(losses)}")
print("-" * 30)
# --- End Debug ---


# --- SAVE THE RESULTS ---
# Simplified the condition slightly for robustness
if train_data and test_data and (test_preds is not None) and losses:
    print("\nSaving results needed for plotting...")
    try:
        torch.save(train_data.y, 'train_labels.pt')
        torch.save(test_data.y, 'test_labels.pt')
        torch.save(test_preds, 'test_predictions.pt')
        torch.save(losses, 'losses.pt')
        print("Results saved successfully to .pt files.")
    except Exception as e:
        print(f"\nError saving files: {e}")

else:
    print("\nTraining pipeline seemed to complete, but one or more variables needed for plotting are invalid. Cannot save plot data.")
    # Print again to show *why* it failed
    print(f"train_data is valid: {train_data is not None and hasattr(train_data, 'y')}")
    print(f"test_data is valid: {test_data is not None and hasattr(test_data, 'y')}")
    print(f"test_preds is valid: {test_preds is not None and isinstance(test_preds, torch.Tensor)}")
    print(f"losses list has items: {bool(losses)}")

Loading embedding model (this may take a moment)...
Embedding model loaded.
--- STARTING MASTER TRAINING PIPELINE ---
Found 242 total contracts.
Creating training set with 193 contracts.
Creating test set with 49 contracts.
Processing (Train): contract_29.pdf

--- Processing: contract_29.pdf ---
Found 41 nodes (clauses/paragraphs).
Created 41 labels. Found 0 risky nodes.
Generating 41 embeddings...


Batches: 100%|██████████| 2/2 [00:00<00:00,  8.15it/s]


Embeddings generated.
Added 40 STRUCTURAL edges.
Calculating semantic similarities...
Added 6 new SEMANTIC edges.
Finding and adding REFERENTIAL edges...
Found 0 defined terms.
Added 0 new REFERENTIAL edges.
Graph built: 41 nodes, 46 total edges.

Converted NetworkX graph to PyTorch Geometric data object.
Processing (Train): contract_177.pdf

--- Processing: contract_177.pdf ---
Found 900 nodes (clauses/paragraphs).
Created 900 labels. Found 45 risky nodes.
Generating 900 embeddings...


Batches: 100%|██████████| 29/29 [00:12<00:00,  2.39it/s]


Embeddings generated.
Added 899 STRUCTURAL edges.
Calculating semantic similarities...
Added 504 new SEMANTIC edges.
Finding and adding REFERENTIAL edges...
Found 5 defined terms.
Added 1 new REFERENTIAL edges.
Graph built: 900 nodes, 1370 total edges.

Converted NetworkX graph to PyTorch Geometric data object.
Processing (Train): contract_155.pdf

--- Processing: contract_155.pdf ---
Found 4 nodes (clauses/paragraphs).
Created 4 labels. Found 0 risky nodes.
Generating 4 embeddings...


...

Embeddings generated.
Added 6 STRUCTURAL edges.
Calculating semantic similarities...
Added 0 new SEMANTIC edges.
Finding and adding REFERENTIAL edges...
Found 0 defined terms.
Added 0 new REFERENTIAL edges.
Graph built: 7 nodes, 6 total edges.

Converted NetworkX graph to PyTorch Geometric data object.
Processing (Test): contract_13.pdf

--- Processing: contract_13.pdf ---
Found 28 nodes (clauses/paragraphs).
Created 28 labels. Found 0 risky nodes.
Generating 28 embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.03it/s]


Embeddings generated.
Added 27 STRUCTURAL edges.
Calculating semantic similarities...
Added 7 new SEMANTIC edges.
Finding and adding REFERENTIAL edges...
Found 0 defined terms.
Added 0 new REFERENTIAL edges.
Graph built: 28 nodes, 34 total edges.

Converted NetworkX graph to PyTorch Geometric data object.
Processing (Test): contract_169.pdf

--- Processing: contract_169.pdf ---
Found 40 nodes (clauses/paragraphs).
Created 40 labels. Found 0 risky nodes.
Generating 40 embeddings...


Batches: 100%|██████████| 2/2 [00:00<00:00,  9.48it/s]


Embeddings generated.
Added 39 STRUCTURAL edges.
Calculating semantic similarities...
Added 4 new SEMANTIC edges.
Finding and adding REFERENTIAL edges...
Found 0 defined terms.
Added 0 new REFERENTIAL edges.
Graph built: 40 nodes, 42 total edges.

Converted NetworkX graph to PyTorch Geometric data object.

Training on 23272 nodes across 193 contracts.
Training with Class Weights: tensor([ 0.3392, 68.6490, 26.3855])
--- Starting Model Training ---
Epoch 000 | Loss: 1.1083
Epoch 020 | Loss: 0.7615
Epoch 040 | Loss: 0.6959
Epoch 060 | Loss: 0.7252
Epoch 080 | Loss: 0.6943
Epoch 100 | Loss: 0.6945
Epoch 120 | Loss: 0.7041
Epoch 140 | Loss: 0.7150
Epoch 160 | Loss: 0.6789
Epoch 180 | Loss: 0.6799
Epoch 200 | Loss: 0.6690
--- Training Complete ---

--- EVALUATING ON TRAIN DATA (MASTER BATCH) ---
Node | Ground Truth Label | Model's Prediction
----------------------------------------
 066 |         1          |         1       [CORRECT]
 069 |         2          |         2       [CORRECT]
 07